In [36]:
import json 
import os  
import pandas as pd

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "poti2010searching")
original_data_pathway = os.path.join(pathway, "original_data")
original_sub_pathway = os.path.join(original_data_pathway, "EVApe_landmark_data_PKanngiesser")

starting_point= original_sub_pathway

temp = []

for dirpath, dirnames, filenames in os.walk(starting_point):
    for index, filename in enumerate([f for f in filenames if f.endswith("raw.csv")]):
        sav_filepath = os.path.join(dirpath, filename)
        # print(sav_filepath)
        x = pd.read_csv(sav_filepath)
        x.columns = map(str.lower, x.columns)
        x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
        x['file_name']= filename

        temp.append(x)

fulldf = pd.concat(temp, ignore_index=True, sort=False)


comp_out_path_stand = os.path.join(original_data_pathway, 'poti_full_raw.csv')
fulldf.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

complete_path_1 = os.path.join(original_data_pathway, "poti_full_raw.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [37]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df.columns = df.columns.str.replace(". ", "_", regex=True)
df['study_id']='poti2010searching'
# df.columns

In [38]:
df.rename(columns={"study": "experiment_name",
   'tria_no.':'trial',
   'sessio_type':'session_type',
   'rewar_position':'reward_position',
   'name':'ape'}, inplace=True)

In [39]:
# df[['year', 'month', 'day_temp']] = df['date'].str.split('-',expand=True)
datedf=[]
for index, row in df.iterrows():
    if "/" in str(row['date']):
        month,day,year = str(row['date']).split('/')
        datedf.append([month,day,year])
    else:
        # print(str(row['date']).split('.'))except
        try:
            day,month,year = str(row['date']).split('.')
            datedf.append([month,day,year])
        except:
            # print(str(row['date']))
            datedf.append(["","",""])

df[[ "month", "day", "year"]] = datedf
# df[['day', 'day_seconds']] = df['day_temp'].str.split(' ',expand=True)

In [40]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [41]:
# df.columns
# df['experiment_name'].unique()

In [42]:
code_list=["experiment_name"]
for index, x in enumerate(code_list):    
    df[x] = df[x].astype(str)
    temp=[]
    for entry in df[x]:
        if entry == '4 landmarks':
            entry = "1b"
        elif entry =='2 landmarks 1 hole':
            entry = "2b"
        elif entry =='2 landmarks 3 holes':
            entry = "2c"
        elif entry =='2 landmarks vd':
            entry = "3b"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col': 'experiment'})


In [43]:
code_list=["session_type"]
for index, x in enumerate(code_list):  
    temp=[]
    for entry in df[x]:
        if entry == 1:
            entry = "training_trial"
        elif entry ==2:
            entry = "control_trial"
        elif entry ==3:
            entry = "expansion_trial"
        temp.append(entry)
    df = df.assign(temp_col=temp)
    df=df.rename(columns={'temp_col':'trial_type'})

In [44]:
# df.columns

In [45]:
choice_list = df[['reward_position','1_choice','2_choice', '3_choice', '4_choice', '5_choice', '6_choice', '7_choice',
       '8_choice', '9_choice', '10_choice', '11_choice', '12_choice',
       '13_choice', '14_choice', '15_choice', '16_choice', '17_choice',
       '18_choice', '19_choice', '20_choice', '21_choice', '22_choice',
       '23_choice', '24_choice', '25_choice', '26_choice', '27_choice',
       '28_choice', '29_choice', '29_choice', '30_choice', '31_choice', '32_choice', '33_choice',
       '34_choice', '35_choice', '36_choice', '37_choice', '38_choice',
       '39_choice', '40_choice']].values.tolist()

middle_of_configuration_found = []
for row in choice_list: 
    if row[0] in row[1:]:
        value = "yes"
    else:
        value = "no"
    middle_of_configuration_found.append(value)

df = df.assign(middle_of_configuration_found=middle_of_configuration_found)
df.columns
df.rename(columns={"ape": "participant",
                    "reward":"reward_present"}, inplace=True)

In [46]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 

df= df.merge(ape_dob,left_on='participant', right_on='name', how='left').copy() #insert dob of participants
df['dodc'] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
df['dodc'].replace("--", np.nan, inplace=True, regex=True)
# df.loc[df.experiment == '3b', ['dodc']] = np.nan
df['dodc'] = pd.to_datetime(df['dodc'])##convert date of data collection to datetime format
df['dob'] = pd.to_datetime(df['dob'])##convert date of birth to datetime format

df['age_in_years'] = (df['dodc'] - df['dob']).dt.days//365


In [47]:
alpha_numeric = os.path.join(original_data_pathway, "poti_coding_schematic_translation.csv")
df_recode  = pd.read_csv(alpha_numeric)

recode_list = ['reward_position','1_choice',
       '2_choice', '3_choice', '6_choice', '7_choice',
       '8_choice', '9_choice', '10_choice', '11_choice', '12_choice',
       '13_choice', '14_choice', '15_choice', '16_choice', '17_choice',
       '18_choice', '19_choice', '20_choice', '21_choice', '22_choice',
       '23_choice', '24_choice', '25_choice', '26_choice', '27_choice',
       '28_choice', '29_choice']


for x,y in zip(df_recode['coding_schematic_for_board_positions'],df_recode['alpha_numeric']):
    for k in recode_list:
       df[k].replace(x, y, inplace=True)


In [48]:
df_recode['coding_schematic_for_board_positions'] = df_recode['coding_schematic_for_board_positions'].astype(str)
df_recode['coding_schematic_for_board_positions'] = df_recode['coding_schematic_for_board_positions'].str.rstrip('.0')

for x,y in zip(df_recode['coding_schematic_for_board_positions'],df_recode['alpha_numeric']):
    df['4_choice'] = df['4_choice'].str.rstrip('.0')
    df['4_choice'].replace(x, y, inplace=True)
# df['4_choice'].unique()

for x,y in zip(df_recode['coding_schematic_for_board_positions'],df_recode['alpha_numeric']):
    df['5_choice'] = df['5_choice'].str.rstrip('.0')
    df['5_choice'].replace(x, y, inplace=True)

In [49]:
# replace_list = ['experiment_name', '4_choice', '5_choice','distance']
# for x in replace_list:
#     df[x].replace(' ', '_', inplace=True, regex=True)

In [50]:
df.dropna(subset=['participant'], inplace=True)


In [51]:

df['experiment'].unique()
# date_list = [['1b','2007'],
#              ['2b','2007'],
#              ['2c','2007'],
#              ['3b','2008']]
# for x,y in date_list:
#     df.loc[df.experiment == x, ['year']] = 7

In [52]:
df=df[[ 'study_id', 'experiment', 'experiment_name', 'year', 'month', 'day', 
        'participant', 'age_in_years','sex', 'species', 'session',
       'trial', 'condition', 'trial_type', 'reward_position', 'reward_present','middle_of_configuration_found',
       '1_choice',
       '2_choice', '3_choice', '4_choice', '5_choice', '6_choice', '7_choice',
       '8_choice', '9_choice', '10_choice', '11_choice', '12_choice',
       '13_choice', '14_choice', '15_choice', '16_choice', '17_choice',
       '18_choice', '19_choice', '20_choice', '21_choice', '22_choice',
       '23_choice', '24_choice', '25_choice', '26_choice', '27_choice',
       '28_choice', '29_choice',  '30_choice', '31_choice',
       '32_choice', '33_choice', '34_choice', '35_choice', '36_choice',
       '37_choice', '38_choice', '39_choice', '40_choice', 'comments']]


In [53]:
df['experiment'].unique()

array(['1b', '2b', '2c', '3b'], dtype=object)

In [54]:
# exp1 = df[df['experiment'] == '1b'] 
# exp2 = df[df['experiment'] == '2b']
# exp3 = df[df['experiment'] == '2c']
# exp4 = df[df['experiment'] == '3b']

# experiments = [[exp1, 'poti2010searching_exp1b'], ##connects df with name of output dataset
#                 [ exp2, 'poti2010searching_exp2b'],
#                 [exp3, 'poti2010searching_exp2c'], 
#                 [exp4, 'poti2010searching_exp3b']]

# for x,y in experiments:
#     x = x.dropna(axis=1, how='all')## drop empty rows/columns
#     comp_out_path_stand = os.path.join(out_pathway, y+'_standardized.csv')
#     x.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)
#     ##glossaries
#     names = x.columns.tolist()
#     df = pd.DataFrame(names)
#     df = df.rename(columns={0: "column_name"})
#     df["description"] = ""
#     studyID_glossary=df[["column_name", "description"]]

#     comp_out_path_glossary = os.path.join(out_pathway, y+'_glossary.csv')
#     studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)


# for index in range(1,5):
#     exp = df[df['experiment'] == str(index)]
#     exp = exp.dropna(axis=1, how='all')
#     comp_out_path = os.path.join(out_pathway, 'poti2010searching_exp'+str(index)+'_standardized.csv')
#     exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
#     names = exp.columns.tolist()
#     exp_g = pd.DataFrame(names)
#     exp_g = exp_g.rename(columns={0: "column_name"})
#     exp_g["description"] = ""
#     exp_g=exp_g[["column_name", "description"]]
#     comp_out_path_glossary = os.path.join(out_pathway, 'poti2010searching_exp'+str(index)+'_glossary.csv')
#     exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)